# Train the Review Router models (Colab)

Runs the **same code** as `train_all.py` in the repo — this notebook holds no
training logic of its own, so results match a local GPU run.

**Runtime → Change runtime type → T4 GPU** before running anything.

Expect roughly **3 hours** on a T4, almost all of it review sentiment. Keep the
tab open; free Colab disconnects idle sessions.

**If Colab disconnects:** reconnect and run every cell again from the top. With
checkpoints on Google Drive (step 6) the corpus is rebuilt identically and
training resumes where it stopped instead of starting over.


## 1. Check the GPU

Stop here if this reports no GPU — the pipeline requires CUDA.

In [ ]:
!nvidia-smi

## 2. Clone the repository

For a private repo, replace the URL with one containing your token.

In [ ]:
REPO_URL = "https://github.com/Sahil-Khilari/intelligent-review-router.git"
BRANCH   = "feat/real-absa-pipeline"

!git clone --branch {BRANCH} {REPO_URL} repo
%cd repo
!git log --oneline -1

## 3. Install dependencies

Colab ships a CUDA torch already, so torch is left alone.

In [ ]:
!pip install -q "transformers>=5.17.0" "datasets>=5.0.1" "accelerate>=1.15.0"     "sentence-transformers>=3.0.0" "seqeval>=1.2.2" "sentencepiece>=0.2.0"     "tiktoken>=0.8.0" "scikit-learn>=1.9.1"

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("gpu  :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

## 4. Hugging Face login

Needed to publish the trained models so your laptop can download them later.
Create a **write** token at https://huggingface.co/settings/tokens.

The token is entered interactively and is never stored in the repo.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
HF_USERNAME = ""  # <-- your Hugging Face username
assert HF_USERNAME, "Set HF_USERNAME before continuing"
print("will publish to:", [f"{HF_USERNAME}/review-router-{t}" for t in ("ate","apc","sentiment")])

## 5. Smoke test first

About two minutes on tiny data. If this fails, the full run would have failed
too. Smoke models go to `models/_smoke/` and are never published or served.

In [ ]:
!python data/fetch_real_datasets.py --profile smoke
!python train_all.py --profile smoke

## 6. Build the real corpora

SemEval-2014 is small. The Amazon stream is the slow part; it stops as soon as
the per-class quota is filled.

Train/test overlap is asserted empty and will abort here if violated.

In [ ]:
!python data/fetch_real_datasets.py --profile t4

## 6b. Keep checkpoints on Google Drive (recommended)

Training saves a checkpoint every epoch. On the VM's own disk those vanish with
a disconnect; on Drive, re-running the training cell resumes from the last one.
Checkpoints are tied to the exact configuration, so this can never resume a
smoke run into the real one.

In [ ]:
USE_DRIVE_CHECKPOINTS = True

import os
if USE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["CHECKPOINT_DIR"] = "/content/drive/MyDrive/review-router-checkpoints"
print("checkpoints ->", os.environ.get("CHECKPOINT_DIR", "checkpoints/ on the VM (lost on disconnect)"))

## 7. Train everything

Three models, sequentially: aspect extraction, aspect polarity, review
sentiment. Each is independent — if one fails the others still finish, and the
failure is reported in the summary.

Interrupted? Run this cell again (after the cells above); it resumes.

In [ ]:
!python train_all.py --profile t4 --push-to-user {HF_USERNAME}

## 8. Results

In [ ]:
import json
print(json.dumps(json.load(open("models/training_summary.json")), indent=2))

## 9. Backup copy (optional)

The models are already on the Hub. This is only if you want a local zip too.

In [ ]:
!zip -qr models.zip models/ -x "models/_smoke/*"
from google.colab import files
files.download("models.zip")

## Done

On your laptop, from the repository folder:

```bash
uv sync --group dev
uv run python scripts/download_models.py --user <HF_USERNAME>
uv run uvicorn main:app --port 8000
curl http://127.0.0.1:8000/health    # expect "degraded": false
```
